# B2 — LoRA Fine-Tuning

## Overview

This notebook performs LoRA fine-tuning for aspect-conditioned sentiment classification as part of the Heard project.

Using the instruction-formatted training and validation datasets prepared in B1, we will fine-tune Qwen2.5-0.5B-Instruct to classify employee review sentiment as positive, neutral, or negative for a specified aspect.

The notebook includes:

1. GPU environment setup
2. Dataset loading and inspection
3. Base model and tokenizer setup
4. LoRA configuration
5. Model fine-tuning
6. Model saving and inference testing

**Input datasets:**
- `train_instructions.jsonl` (18,000 samples)
- `validation_instructions.jsonl` (1,800 samples)

**Expected output:**
- Fine-tuned LoRA adapter
- Training and validation metrics
- Inference examples

The fine-tuned model will be evaluated against the base model in B3.

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


## 1. Environment Setup

Configure the GPU environment and install the libraries required for LoRA fine-tuning.

In [2]:
# Install required libraries
%pip install -q transformers datasets peft trl accelerate

In [3]:
# Check installed library versions
import transformers
import datasets
import peft
import trl
import accelerate

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.22.1
Datasets: 4.8.5
PEFT: 0.21.2
TRL: 1.17.1
Accelerate: 1.15.0


## 2. Dataset Preparation

Load the instruction-formatted training and validation datasets prepared in B1 for LoRA fine-tuning.

In [4]:
from datasets import load_dataset

# Load training and validation datasets
dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train_instructions.jsonl",
        "validation": "/content/validation_instructions.jsonl"
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 18000
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1800
    })
})


In [5]:
# Inspect one training example
print(dataset["train"][0])

{'instruction': 'Analyze the employee review and determine the sentiment for the specified aspect. Choose one of: positive, neutral, negative. Return only a JSON object.', 'input': "Aspect: culture_values\nReview: TITLE: great job PROS: team collaboration, fun working environment, good opportunities CONS: can be a bit dangerous depending on the area you're working in", 'output': '{"sentiment": "positive"}'}


## 3. Base Model & LoRA Configuration

Load Qwen2.5-0.5B-Instruct and configure the model for parameter-efficient fine-tuning using LoRA.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Define base model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", model.device)
print("Parameters:", f"{model.num_parameters():,}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0
Parameters: 494,032,768


### LoRA Configuration

LoRA (Low-Rank Adaptation) is used to fine-tune the Qwen2.5-0.5B-Instruct model efficiently.

Instead of updating all model parameters, LoRA introduces trainable low-rank matrices into selected attention layers.

The configuration uses:

- Rank (r): 8
- Alpha: 16
- Dropout: 0.05
- Target modules: q_proj and v_proj
- Task type: Causal Language Modeling

This approach reduces the number of trainable parameters and allows fine-tuning on limited GPU resources.

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to the base model
model = get_peft_model(model, lora_config)

# Check trainable parameters
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 4. Data Formatting and Tokenization

Prepare the instruction-formatted datasets for supervised fine-tuning.

Each example contains a system instruction, an employee review with a specified aspect, and the expected sentiment output.

The data is formatted using the Qwen chat template and tokenized for model training.

In [8]:
# Format instruction data using the model's chat template
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": example["instruction"]
        },
        {
            "role": "user",
            "content": example["input"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

# Apply formatting to training and validation datasets
formatted_dataset = dataset.map(format_example)

print(formatted_dataset)

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 18000
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 1800
    })
})


In [9]:
# Inspect one formatted example
print(formatted_dataset["train"][0]["text"])

<|im_start|>system
Analyze the employee review and determine the sentiment for the specified aspect. Choose one of: positive, neutral, negative. Return only a JSON object.<|im_end|>
<|im_start|>user
Aspect: culture_values
Review: TITLE: great job PROS: team collaboration, fun working environment, good opportunities CONS: can be a bit dangerous depending on the area you're working in<|im_end|>
<|im_start|>assistant
{"sentiment": "positive"}<|im_end|>



In [10]:
from transformers import TrainingArguments

# Configure training arguments for a small test run
training_args = TrainingArguments(
    output_dir="/content/lora_test",

    num_train_epochs=1,
    max_steps=20,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,

    fp16=True,

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",

    report_to="none"
)

print("Training arguments configured successfully!")

Training arguments configured successfully!


In [12]:
# Tokenize examples and mask non-assistant tokens
MAX_LENGTH = 512

def tokenize_example(example):

    # System and user messages
    prompt_messages = [
        {
            "role": "system",
            "content": example["instruction"]
        },
        {
            "role": "user",
            "content": example["input"]
        }
    ]

    # Full conversation
    full_messages = prompt_messages + [
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    # Convert messages to text
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize text
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    # Truncate to maximum length
    input_ids = full_ids[:MAX_LENGTH]

    # Mask prompt tokens
    prompt_length = len(prompt_ids)

    labels = [
        -100 if i < prompt_length else token_id
        for i, token_id in enumerate(input_ids)
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }


# Apply tokenization
tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 18000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1800
    })
})


In [13]:
# Check tokenization and label masking
sample = tokenized_dataset["train"][0]

print("Input length:", len(sample["input_ids"]))
print("Label length:", len(sample["labels"]))

valid_labels = [
    label for label in sample["labels"]
    if label != -100
]

print("Number of valid labels:", len(valid_labels))

print(
    "Assistant target:",
    tokenizer.decode(valid_labels)
)

Input length: 90
Label length: 90
Number of valid labels: 9
Assistant target: {"sentiment": "positive"}<|im_end|>



### Tokenization and Label Masking Validation

Validate the tokenized training and validation datasets before fine-tuning.

Prompt tokens are masked with -100 so that the training loss is calculated only on the assistant response.

Samples without valid training labels are removed. Remaining samples reaching the maximum sequence length are inspected to identify potentially truncated responses.

In [14]:
# Check label masking across the full dataset

for split in ["train", "validation"]:

    total = len(tokenized_dataset[split])

    no_target = 0
    truncated = 0

    for sample in tokenized_dataset[split]:

        labels = sample["labels"]

        # Check if all labels are masked
        if all(label == -100 for label in labels):
            no_target += 1

        # Check if sequence reaches maximum length
        if len(sample["input_ids"]) == MAX_LENGTH:
            truncated += 1

    print(f"{split}:")
    print("Total samples:", total)
    print("Samples without target:", no_target)
    print("Samples reaching max length:", truncated)
    print()

train:
Total samples: 18000
Samples without target: 86
Samples reaching max length: 88

validation:
Total samples: 1800
Samples without target: 9
Samples reaching max length: 9



In [15]:
# Remove samples without valid training labels

def has_valid_labels(example):
    return any(label != -100 for label in example["labels"])

tokenized_dataset = tokenized_dataset.filter(has_valid_labels)

print(tokenized_dataset)

Filter:   0%|          | 0/18000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 17914
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1791
    })
})


In [16]:
# Check remaining samples reaching maximum length

for split in ["train", "validation"]:

    max_length_samples = [
        sample
        for sample in tokenized_dataset[split]
        if len(sample["input_ids"]) == MAX_LENGTH
    ]

    print(f"{split}:")
    print("Samples reaching max length:", len(max_length_samples))

    for sample in max_length_samples[:5]:

        valid_labels = [
            label for label in sample["labels"]
            if label != -100
        ]

        print("Target:", tokenizer.decode(valid_labels))

train:
Samples reaching max length: 2
Target: {"sentiment":
Target: {"sent
validation:
Samples reaching max length: 0


In [17]:
# Remove remaining truncated samples

tokenized_dataset["train"] = tokenized_dataset["train"].filter(
    lambda example: len(example["input_ids"]) < MAX_LENGTH
)

print(tokenized_dataset)

Filter:   0%|          | 0/17914 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 17912
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1791
    })
})


## 5. Initial LoRA Fine-Tuning Test

Before full training, a small-scale experiment is conducted using 20 training steps.

This test verifies that the training pipeline runs successfully and that the model can generate sentiment predictions.

Training and validation losses are monitored to assess the initial training behavior.

In [18]:
# Configure data collator

import torch

def data_collator(features):

    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for f in features:

        pad_len = max_len - len(f["input_ids"])

        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_len
        )

        attention_masks.append(
            f["attention_mask"] + [0] * pad_len
        )

        labels.append(
            f["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

print("Data collator configured successfully!")

Data collator configured successfully!


In [19]:
# Initialize Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

print("Trainer initialized successfully!")

Trainer initialized successfully!


In [20]:
# Run a 20-step LoRA fine-tuning test

train_result = trainer.train()

print("Training completed successfully!")
print(train_result)

Step,Training Loss,Validation Loss
10,0.145893,0.140948
20,0.123571,0.124941


Training completed successfully!
TrainOutput(global_step=20, training_loss=0.18292252123355865, metrics={'train_runtime': 114.5629, 'train_samples_per_second': 1.397, 'train_steps_per_second': 0.175, 'total_flos': 48384934574592.0, 'train_loss': 0.18292252123355865, 'epoch': 0.008932559178204555})


In [21]:
# Check training results

print("Training metrics:")
print(train_result.metrics)

print("\nEvaluation metrics:")
eval_results = trainer.evaluate()
print(eval_results)

Training metrics:
{'train_runtime': 114.5629, 'train_samples_per_second': 1.397, 'train_steps_per_second': 0.175, 'total_flos': 48384934574592.0, 'train_loss': 0.18292252123355865, 'epoch': 0.008932559178204555}

Evaluation metrics:


Training Loss,Validation Loss,Step
0.123571,0.124941,20


{'eval_loss': 0.1249411553144455}


In [22]:
# Save LoRA adapter from the test run

test_adapter_path = "/content/lora_test_adapter"

trainer.save_model(test_adapter_path)

tokenizer.save_pretrained(test_adapter_path)

print("Test LoRA adapter saved successfully!")
print("Saved to:", test_adapter_path)

Test LoRA adapter saved successfully!
Saved to: /content/lora_test_adapter


In [23]:
# Test inference with the fine-tuned LoRA model

import torch

model.eval()

def predict_sentiment(aspect, review):

    instruction = (
        "Analyze the employee review and determine the sentiment "
        "for the specified aspect. "
        "Choose one of: positive, neutral, negative. "
        "Return only a JSON object."
    )

    messages = [
        {
            "role": "system",
            "content": instruction
        },
        {
            "role": "user",
            "content": f"Aspect: {aspect}\nReview: {review}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return prediction.strip()

### Initial Test Results

The initial 20-step training run completed successfully.

The model was tested using a review containing both positive and negative sentiments.

Results:

| Aspect | Expected | Predicted |
|---|---|---|
| comp_benefits | positive | negative |
| senior_mgmt | negative | negative |

The model correctly identified the negative sentiment toward senior management but failed to identify the positive sentiment toward compensation and benefits.

These results suggest that additional training may improve aspect-specific sentiment predictions.

A full fine-tuning run is conducted next.

In [24]:
# Test aspect-based sentiment predictions

test_review = (
    "The salary and benefits are excellent, "
    "but senior management is terrible."
)

test_cases = [
    ("comp_benefits", "positive"),
    ("senior_mgmt", "negative")
]

for aspect, expected in test_cases:

    prediction = predict_sentiment(
        aspect,
        test_review
    )

    print("Aspect:", aspect)
    print("Expected:", expected)
    print("Predicted:", prediction)
    print("-" * 50)

Aspect: comp_benefits
Expected: positive
Predicted: {"sentiment": "negative"}
--------------------------------------------------
Aspect: senior_mgmt
Expected: negative
Predicted: {"sentiment": "negative"}
--------------------------------------------------


In [25]:
# Release the test model before full fine-tuning

import gc
import torch

del trainer
del model

gc.collect()
torch.cuda.empty_cache()

print("Test model released successfully!")

Test model released successfully!


In [26]:
# Check available GPU memory

print("GPU:", torch.cuda.get_device_name(0))

free_memory, total_memory = torch.cuda.mem_get_info()

print(f"Free GPU memory: {free_memory / 1024**3:.2f} GB")
print(f"Total GPU memory: {total_memory / 1024**3:.2f} GB")

GPU: Tesla T4
Free GPU memory: 13.48 GB
Total GPU memory: 14.56 GB


## 6. Full LoRA Fine-Tuning

After the initial test, the original Qwen2.5-0.5B-Instruct model is reloaded with a fresh LoRA adapter.

Full fine-tuning is performed using the prepared training dataset.

Training configuration:

- Training samples: 17,912
- Validation samples: 1,791
- Epochs: 1
- Batch size: 2
- Gradient accumulation steps: 4
- Learning rate: 2e-4
- Weight decay: 0.01
- Warmup steps: 67
- Precision: FP16

Training and validation losses are monitored throughout the process.

In [27]:
# Reload the original base model for full fine-tuning

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Load the original base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply a fresh LoRA adapter
model = get_peft_model(model, lora_config)

# Check trainable parameters
model.print_trainable_parameters()

print("Base model and fresh LoRA adapter loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
Base model and fresh LoRA adapter loaded successfully!


In [29]:
from transformers import TrainingArguments

# Configure training arguments for full LoRA fine-tuning
full_training_args = TrainingArguments(
    output_dir="/content/lora_full",

    num_train_epochs=1,
    max_steps=-1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=67,

    fp16=True,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    report_to="none"
)

print("Full training arguments configured successfully!")

Full training arguments configured successfully!


In [32]:
from transformers import Trainer, DataCollatorForSeq2Seq

# Configure data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

# Create trainer for full LoRA fine-tuning
full_trainer = Trainer(
    model=model,
    args=full_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

print("Full Trainer initialized successfully!")
print("Training samples:", len(tokenized_dataset["train"]))
print("Validation samples:", len(tokenized_dataset["validation"]))

Full Trainer initialized successfully!
Training samples: 17912
Validation samples: 1791


In [33]:
# Check full training configuration

print("Max steps:", full_training_args.max_steps)
print("Epochs:", full_training_args.num_train_epochs)
print("Batch size:", full_training_args.per_device_train_batch_size)
print("Gradient accumulation:", full_training_args.gradient_accumulation_steps)

print("Evaluation strategy:", full_training_args.eval_strategy)
print("Evaluation steps:", full_training_args.eval_steps)

print("Save strategy:", full_training_args.save_strategy)
print("Save steps:", full_training_args.save_steps)

Max steps: -1
Epochs: 1
Batch size: 2
Gradient accumulation: 4
Evaluation strategy: IntervalStrategy.STEPS
Evaluation steps: 500
Save strategy: SaveStrategy.STEPS
Save steps: 500


### Full Training Results

The full LoRA fine-tuning process completed successfully after 2,239 training steps.

Training and validation losses decreased throughout the training process.

| Step | Training Loss | Validation Loss |
|---|---|---|
| 500 | 0.102757 | 0.106234 |
| 1000 | 0.098422 | 0.097682 |
| 1500 | 0.097716 | 0.095459 |
| 2000 | 0.096456 | 0.094882 |
| 2239 | 0.096513 | 0.094868 |

The final validation loss was approximately 0.0949.

The decreasing losses indicate that the model improved its ability to predict the target sentiment responses.

However, training loss and validation loss do not directly measure classification accuracy.

Formal evaluation using a manually labeled test set and Macro-F1 will be conducted in B3.

In [34]:
# Start full LoRA fine-tuning

full_train_result = full_trainer.train()

print("Full LoRA training completed successfully!")
print(full_train_result)

Step,Training Loss,Validation Loss
500,0.102757,0.106234
1000,0.098422,0.097682
1500,0.097716,0.095459
2000,0.096456,0.094882
2239,0.096513,0.094868


Full LoRA training completed successfully!
TrainOutput(global_step=2239, training_loss=0.10338377963224124, metrics={'train_runtime': 1488.8023, 'train_samples_per_second': 12.031, 'train_steps_per_second': 1.504, 'total_flos': 5225155711311360.0, 'train_loss': 0.10338377963224124, 'epoch': 1.0})


## 7. Model Saving

After full fine-tuning, the trained LoRA adapter is saved separately from the base model.

The saved adapter includes the trained LoRA weights, configuration files, and tokenizer files.

The adapter can later be loaded together with the original Qwen base model for inference without repeating the training process.

In [35]:
# Save the final LoRA adapter

final_adapter_path = "/content/lora_final_adapter"

full_trainer.save_model(final_adapter_path)

tokenizer.save_pretrained(final_adapter_path)

print("Final LoRA adapter saved successfully!")
print("Saved to:", final_adapter_path)

Final LoRA adapter saved successfully!
Saved to: /content/lora_final_adapter


In [36]:
import os

print("Saved files:")

for file in os.listdir(final_adapter_path):
    print(file)

Saved files:
chat_template.jinja
README.md
training_args.bin
tokenizer.json
adapter_config.json
adapter_model.safetensors
tokenizer_config.json


## 8. Final Inference Testing

The fully fine-tuned model is tested on the same mixed-sentiment review used during the initial experiment.

Results:

| Aspect | Expected | Predicted |
|---|---|---|
| comp_benefits | positive | positive |
| senior_mgmt | negative | negative |

Both predictions are correct.

Compared with the initial 20-step experiment, the full fine-tuning run successfully identifies the positive sentiment toward compensation and benefits.

This provides preliminary evidence that additional fine-tuning improved the model's performance on this example.

However, these two examples are insufficient to establish overall classification accuracy.

In [37]:
# Test the fully fine-tuned model

model.eval()

test_review = (
    "The salary and benefits are excellent, "
    "but senior management is terrible."
)

test_cases = [
    ("comp_benefits", "positive"),
    ("senior_mgmt", "negative")
]

for aspect, expected in test_cases:

    prediction = predict_sentiment(
        aspect,
        test_review
    )

    print("Aspect:", aspect)
    print("Expected:", expected)
    print("Predicted:", prediction)
    print("-" * 50)

Aspect: comp_benefits
Expected: positive
Predicted: {"sentiment": "positive"}
--------------------------------------------------
Aspect: senior_mgmt
Expected: negative
Predicted: {"sentiment": "negative"}
--------------------------------------------------


## 9. Model Export and Deliverables

The final LoRA adapter is compressed and exported for integration with other project modules.

The deliverables include:

- Fine-tuned LoRA adapter
- Training notebook
- Inference script
- Model documentation

The inference script allows other team members to load the trained adapter and generate aspect-based sentiment predictions.

The adapter must be used together with the corresponding Qwen base model.

In [38]:
# Compress the final LoRA adapter

import shutil

shutil.make_archive(
    "/content/lora_final_adapter",
    "zip",
    "/content/lora_final_adapter"
)

print("Adapter compressed successfully!")

Adapter compressed successfully!


In [39]:
# Download the final LoRA adapter

from google.colab import files

files.download("/content/lora_final_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Independent Model Loading

An independent inference pipeline is prepared to support model reuse outside the training notebook.

The pipeline loads the original Qwen base model, applies the trained LoRA adapter, and generates sentiment predictions.

This enables integration with the project's API and downstream applications.

Independent loading and inference should be verified before final delivery.

In [40]:
# Load the fine-tuned LoRA model for inference

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

adapter_path = "/content/lora_final_adapter"

# Read the exact base model name from the saved adapter
from peft import PeftConfig

peft_config = PeftConfig.from_pretrained(adapter_path)
base_model_name = peft_config.base_model_name_or_path

print("Base model:", base_model_name)

# Load tokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(
    adapter_path
)

# Load base model
inference_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load trained LoRA adapter
inference_model = PeftModel.from_pretrained(
    inference_base_model,
    adapter_path
)

inference_model.eval()

print("Fine-tuned model loaded successfully!")

Base model: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!


In [41]:
%%writefile /content/inference.py

import json
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig


class SentimentPredictor:

    def __init__(self, adapter_path):

        config = PeftConfig.from_pretrained(adapter_path)
        base_model_name = config.base_model_name_or_path

        self.tokenizer = AutoTokenizer.from_pretrained(
            adapter_path
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        self.model = PeftModel.from_pretrained(
            base_model,
            adapter_path
        )

        self.model.eval()

    def predict(self, aspect, review):

        messages = [
            {
                "role": "system",
                "content": (
                    "Analyze the employee review and determine "
                    "the sentiment for the specified aspect. "
                    "Choose one of: positive, neutral, negative. "
                    "Return only a JSON object."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Aspect: {aspect}\n"
                    f"Review: {review}"
                )
            }
        ]

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        prediction = self.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        try:
            result = json.loads(prediction)

            if result.get("sentiment") not in [
                "positive",
                "neutral",
                "negative"
            ]:
                return {"sentiment": None, "raw_output": prediction}

            return result

        except json.JSONDecodeError:
            return {
                "sentiment": None,
                "raw_output": prediction
            }

Writing /content/inference.py


In [43]:
readme_content = """# Aspect-Based Sentiment Analysis — LoRA Fine-Tuning

## 1. Project Overview

This module performs aspect-based sentiment analysis (ABSA)
on employee reviews using a Qwen language model fine-tuned with LoRA.

Given an employee review and a specified aspect, the model
predicts one of three sentiment labels:

- positive
- neutral
- negative

Example:

Review:
"The salary and benefits are excellent, but senior management is terrible."

Aspect: comp_benefits

Output:
{"sentiment": "positive"}

Aspect: senior_mgmt

Output:
{"sentiment": "negative"}


## 2. Model Architecture

Base Model:
See base_model_name_or_path in adapter_config.json.

Fine-Tuning Method:
LoRA (Low-Rank Adaptation)

LoRA Configuration:
- Rank (r): 8
- Alpha: 16
- Dropout: 0.05
- Target modules: q_proj, v_proj
- Task type: CAUSAL_LM

Only approximately 0.1093% of model parameters were trainable.


## 3. Training Dataset

Training samples: 17,912
Validation samples: 1,791

Training labels were generated using weak supervision
based on aspect ratings.

The model predicts sentiment for a specified aspect
rather than the overall sentiment of the review.


## 4. Training Configuration

Epochs: 1
Training steps: 2,239

Per-device batch size: 2
Gradient accumulation steps: 4

Learning rate: 2e-4
Weight decay: 0.01
Warmup steps: 67

Precision: FP16

Hardware: NVIDIA Tesla T4


## 5. Training Results

Final logged training loss: 0.096513
Final validation loss: 0.094868

These losses measure language-model prediction performance
on the training and validation datasets.

They do not directly represent classification accuracy.

Formal evaluation using a manually labeled test set
and Macro-F1 is required.


## 6. Model Inference

Install dependencies:

pip install torch transformers peft accelerate

Example usage:

from inference import SentimentPredictor

predictor = SentimentPredictor("./lora_final_adapter")

result = predictor.predict(
    aspect="comp_benefits",
    review="The salary is excellent, but management is terrible."
)

print(result)


## 7. Model Files

The LoRA adapter is stored in:

lora_final_adapter/

Important files:

- adapter_model.safetensors
- adapter_config.json
- tokenizer.json
- tokenizer_config.json
- chat_template.jinja

The adapter does not contain the full base model weights.


## 8. Limitations

- Training labels were generated using weak supervision.
- Weak labels may contain noise.
- The model has not yet been formally evaluated
  against a manually labeled test set.
- Generalization performance remains to be verified.

Next step:
Compare the fine-tuned model with the original base model
using Macro-F1 on held-out human-labeled reviews.
"""

with open("/content/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md created successfully!")

README.md created successfully!


## 10. Summary and Next Steps

This notebook completed the first full LoRA fine-tuning run for aspect-based sentiment analysis.

Key accomplishments:

- Prepared instruction-formatted training data
- Applied tokenization and label masking
- Completed an initial 20-step training test
- Completed full LoRA fine-tuning
- Saved the final LoRA adapter
- Conducted preliminary sentiment prediction tests
- Prepared model inference and documentation files

The final validation loss was approximately 0.0949.

The model correctly classified both aspects in the preliminary mixed-sentiment test.

### Next Steps

In B3, the fine-tuned model will be compared with the original Qwen base model using a manually labeled held-out test set.

Macro-F1 and class-level performance metrics will be used to evaluate the effectiveness of LoRA fine-tuning.